<table style="width:100%">
  <tr>
    <td valign="top"><img src="../data/img/FER_logo_2.png" width=300 height=80 align="left"></td>
    <td valign="top"><img src="../data/img/LARES_2_transparent.png" width=250 height=80 align="right"></td>
  </tr>
 </table>

# Ensemble learning

![](../data/img/orchestra.jpg)

Dataset from: https://www.kaggle.com/c/house-prices-advanced-regression-techniques/data

## Setup

Run this cell first, in every notebook. It fetches the course repository into
the Colab session and moves into the `notebooks/` folder, so that the
`../data/...` paths work.

It is safe to run more than once, and safe after a restart.

Note: outside Colab the cell does nothing except report the working directory.
Start Jupyter from inside `notebooks/` and the paths work the same way.

If it prints `data ok: True`, we are set.

In [ ]:
# --- SETUP: run this first ---
# works in Colab and locally, safe to run more than once
import os, sys, subprocess
REPO = "ai_bootcamp_foundations"
if "google.colab" in sys.modules:
    if not os.path.isdir(f"/content/{REPO}"):
        subprocess.run(["git", "clone", "-q",
                        f"https://github.com/unizg-fer-lares/{REPO}.git"],
                       cwd="/content", check=True)
    os.chdir(f"/content/{REPO}/notebooks")
print("cwd:", os.getcwd(), "| data ok:", os.path.isdir("../data"))

### Import modules

In [ ]:
# import packages for data processing and plotting
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn import preprocessing
from scipy import stats
import pickle
import seaborn as sns
sns.set_theme()
import matplotlib.pyplot as plt

np.random.seed(123)

#%matplotlib inline

### Auxilary functions

In [ ]:
# helper functions

def add_fit_to_histplot(a, fit=stats.norm, ax=None):
    """
    Auxiliary function to plot PDF of equivalent normal distribution,
    since sns.distplot is deprecated.
    """
    if ax is None:
        ax = plt.gca()

    # compute bandwidth
    bw = len(a)**(-1/5) * a.std(ddof=1)
    # initialize PDF support
    x = np.linspace(a.min()-bw*3, a.max()+bw*3, 200)
    # compute PDF parameters
    params = fit.fit(a)
    # compute PDF values
    y = fit.pdf(x, *params)
    # plot the fitted continuous distribution
    ax.plot(x, y, color='#282828')
    return ax

def transform_feat(feat, source, sink, transform):
    """
    Transforms a column in source and places it in sink.
    Prints dataframe with analysis, before and after.
    Plots 2 KDE plots, before and after.
    """
    
    # apply transformation
    sink[feat] = source[feat].apply(transform)
    
    # make analysis
    temp1 = source[feat].describe()
    temp1['skew'] = source[feat].skew()
    temp1['kurt'] = source[feat].kurt()
    temp2 = sink[feat].describe()
    temp2['skew'] = sink[feat].skew()
    temp2['kurt'] = sink[feat].kurt()
    print(pd.DataFrame([temp1, temp2], index=['Before transform', 'After transform']).T)
    
    # plot histogram+KDE
    fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=[14, 4])
    ax1 = sns.histplot(source[feat], stat='density', kde=True, ax=ax1)
    ax1 = add_fit_to_histplot(source[feat], fit=stats.norm, ax=ax1)
    ax1.legend(['Actual distribution', 'Normal distribution'])
    ax1.set_title('Histogram and KDE, before transform')
    ax2 = sns.histplot(sink[feat], stat='density', kde=True, ax=ax2)
    ax2 = add_fit_to_histplot(sink[feat], fit=stats.norm, ax=ax2)
    ax2.legend(['Actual distribution', 'Normal distribution'])
    ax2.set_title('Histogram and KDE, after transform')

def compare_cat(feat, df, figsize, order=None):
    """
    Compares a categorical feature to the label in a boxplot.
    """
    fig, ax = plt.subplots(figsize=figsize)
    ax = sns.boxplot(x=feat, y="SalePrice", hue=feat, legend=False, data=df, 
                     order=order, palette="muted", flierprops={"marker": "x"})
    if len(ax.get_xticks()) > 15:
        ax.xaxis.set_tick_params(rotation=45)
    else:
        ax.xaxis.set_tick_params(rotation=0)
    ax.set_title("Box plot - comperison with label")
    
def compare_num(feat, source, sink):
    """
    Compares a numerical feature to the label in a scatterplot.
    """
    fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=[14, 4])
    ax1 = sns.scatterplot(x=feat, y="SalePrice", data=source, ax=ax1, alpha=0.5)
    ax1.set_title("Scatter plot - before transformations")
    ax2 = sns.scatterplot(x=feat, y="SalePrice", data=sink, ax=ax2, alpha=0.5)
    ax2.set_title("Scatter plot - after transformations");

## 0. (Pre)process data

In [ ]:
# load the dataset
filepath = '../data/housing_prices/housing.csv'
df_raw = pd.read_csv(filepath, index_col='Id')

# take a look at it
print("Shape of dataset:", df_raw.shape)
df_raw.sample(5)

In [ ]:
# choose columns to use
cols = ['MSSubClass', 'LotArea', 'Neighborhood', 'OverallQual', 'YearBuilt', 'PoolArea', 'ExterQual',
        'BsmtUnfSF', 'TotalBsmtSF', 'GrLivArea', 'KitchenQual', 'GarageArea', 'MoSold', 'SalePrice']
df_raw = df_raw[cols]

# check the dataset
print('Shape of dataset:', df_raw.shape)
print('Data types:')
print(df_raw.dtypes)
df_raw.sample(5)

In [ ]:
# check for missing values
print('Number of NaN values per columns:')
print(df_raw.isna().sum())

In [ ]:
# make another dataframe to save the "raw" dataframe
df_proc = df_raw.copy()

In [ ]:
# SalePrice: label, target values
# -> numerical -- regression problem
# -> log transformation to be closer to normal distribution
feat = 'SalePrice'
transform_feat(feat, df_raw, df_proc, np.log1p)

In [ ]:
# MSSubClass: Identifies the type of dwelling involved in the sale.
# -> categorical feature
# -> originally int64, but it's type should be object
# -> it will be one-hot encoded later
feat = 'MSSubClass'
df_proc[feat] = df_proc[feat].astype('object')
print('Original data type of', feat, 'is', df_raw[feat].dtype)
print('New data type of', feat, 'is', df_proc[feat].dtype)
compare_cat(feat, df_proc, (12,4))

In [ ]:
# LotArea: Lot size in square feet
# -> numerical feature
# -> no transformations
feat = 'LotArea'
compare_num(feat, df_raw, df_proc)

In [ ]:
# Neighborhood: Physical locations within Ames city limits
# -> categorical feature
# -> no need for changing
# -> it will be one-hot encoded later
feat = 'Neighborhood'
compare_cat(feat, df_proc, (16,4))

In [ ]:
# OverallQual: Rates the overall material and finish of the house
# -> seems categorical, but it can be treated as numerical feature
# -> no need for processing
feat = 'OverallQual'
compare_cat(feat, df_proc, (6,4))
compare_num(feat, df_raw, df_proc)

In [ ]:
# YearBuilt: Original construction date
# -> numerical feature
# -> subtract the minimal value so that values start with 0
feat = 'YearBuilt'
df_proc[feat] = df_raw[feat] - df_raw[feat].min()
compare_num(feat, df_raw, df_proc)

In [ ]:
# PoolArea: Pool area in square feet
# -> originally numerical feature
# -> since not many houses have a pool, this can be turned into categorical feature:
#    does the house have a pool or not (binary feature)
# -> changed into integer (as one-hot encoding)
feat = 'PoolArea'
df_proc[feat] = (df_raw[feat] > 0).astype(int)
compare_cat(feat, df_proc, (4,4))

In [ ]:
# ExterQual: Evaluates the quality of the material on the exterior
# -> seems categorical, but it can be treated as numerical feature
# -> string marks converted into integers
feat = 'ExterQual'
marks = {'Fa': '1', 'TA': '2', 'Gd': '3', 'Ex': '4'}
df_proc[feat] = df_raw[feat].replace(marks).astype(np.int64)
compare_cat(feat, df_proc, (6,4))
compare_num(feat, df_raw, df_proc)

In [ ]:
# BsmtUnfSF: Unfinished square feet of basement area
# -> numerical feature
# -> no need for transformation
feat = 'BsmtUnfSF'
compare_num(feat, df_raw, df_proc)

In [ ]:
# TotalBsmtSF: Total square feet of basement area
# -> numerical feature
# -> no transformation
feat = 'TotalBsmtSF'
compare_num(feat, df_raw, df_proc)

In [ ]:
# GrLivArea: Above grade (ground) living area square feet
# -> numerical feature
# -> log transformation
feat = 'GrLivArea'
transform_feat(feat, df_raw, df_proc, np.log1p)
compare_num(feat, df_raw, df_proc)

In [ ]:
# KitchenQual: Kitchen quality
# -> seems categorical, but it can be treated as numerical feature
feat = 'KitchenQual'
marks = {'Fa': '1', 'TA': '2', 'Gd': '3', 'Ex': '4'}
df_proc[feat] = df_raw[feat].replace(marks).astype(np.int64)
compare_cat(feat, df_proc, (6,4))
compare_num(feat, df_raw, df_proc)

In [ ]:
# GarageArea: Size of garage in square feet
# -> numerical feature
feat = 'GarageArea'
compare_num(feat, df_raw, df_proc)

In [ ]:
# MoSold: Month Sold (MM)
# -> categorical feature
# -> convert int to object
feat = 'MoSold'
df_proc[feat] = df_raw[feat].astype('object')
compare_cat(feat, df_proc, (10,4))

In [ ]:
# one-hot encoding of categorical features
cat_cols = ['MSSubClass', 'Neighborhood', 'PoolArea', 'MoSold']
df_cat_1hot = pd.get_dummies(df_proc[cat_cols], columns=cat_cols, drop_first=True)

In [ ]:
# scaling numerical features
num_cols = ['LotArea', 'OverallQual', 'YearBuilt', 'ExterQual', 'BsmtUnfSF', 
            'TotalBsmtSF', 'GrLivArea', 'KitchenQual', 'GarageArea']
scaler = preprocessing.RobustScaler()
scaler.fit(df_proc[num_cols])
df_num_scaled = pd.DataFrame(scaler.transform(df_proc[num_cols]), columns=num_cols, index=df_proc.index)

In [ ]:
# merging numerical and categorical features, and label
df_full = pd.concat([df_num_scaled, df_cat_1hot, df_proc['SalePrice'].to_frame()], axis=1)

In [ ]:
# divide data into train and test sets, but keep full data available as well
from sklearn.model_selection import train_test_split

X = df_full.drop(columns='SalePrice')
y = df_full['SalePrice']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)

## 1. Cross-validation & simple grid-search

### K-fold cross-validation for evaluation
<img src="../data/img/Kfold_CV.png" alt="Kfold_CV" width="800"/> \
https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html \
`sklearn.model_selection.`**`cross_val_score`**`(estimator, X, y=None, *, groups=None, scoring=None, cv=None, n_jobs=None, verbose=0, fit_params=None, params=None, pre_dispatch='2*n_jobs', error_score=nan)`

https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html \
`sklearn.model_selection.`**`KFold`**`(n_splits=5, *, shuffle=False, random_state=None)`

In [ ]:
# additional imports
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import root_mean_squared_error

HANDS-ON: Model performance with classical data splitting

In [ ]:
# tune a simple Ridge estimator with default hyperparameter
estimator_classical = ???
???

# generate predictions
y_hat_classical = ???

# calculate and print RMSE
error = ???
print('RMSE = {:.6}'.format(error))

Model performance with cross-validation

In [ ]:
# which K to use?
print('Previous test set uses {:.4}% of data.'.format(X_test.shape[0]/X.shape[0]*100))

In [ ]:
# define model with default hyperparameter
estimator_CV = LinearRegression()

# generate predictions using CV
cv = KFold(n_splits=5, shuffle=True, random_state=33)
errors = cross_val_score(estimator_CV, X, y, scoring='neg_root_mean_squared_error', cv=cv)

# print all errors including mean and std
print('RMSE with CV = ')
for e, f in zip(errors, range(1, 6)):
    print('Fold {}: {:.6}'.format(f, -e))
print('Mean  : {:.6}'.format(-np.array(errors).mean()))
print('Std   : {:.6}'.format(np.array(errors).std()))

### Grid Search cross-validation
https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html \
`sklearn.model_selection.`**`GridSearchCV`**`(estimator, param_grid, *, scoring=None, n_jobs=None, refit=True, cv=None, verbose=0, pre_dispatch='2*n_jobs', error_score=nan, return_train_score=False)`

<img src="../data/img/CV_datasplit.png" alt="CV_datasplit" width="650"/>

In [ ]:
# additional import
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

In [ ]:
# tune a simple Ridge estimator with default hyperparameter for reference
estimator_simple = Ridge()
estimator_simple.fit(X_train, y_train)

# generate predictions
y_hat_simple = estimator_simple.predict(X_test)

# calculate and print RMSE
error = root_mean_squared_error(y_hat_simple, y_test)
print('RMSE = {:6f}'.format(error))

In [ ]:
# grid search with cv on a Ridge estimator
estimator_cv = Ridge()
param_grid1 = {
    'alpha': [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000]
}
gscv1 = GridSearchCV(estimator_cv, param_grid1, cv=5, scoring='neg_root_mean_squared_error', refit=True)
gscv1.fit(X_train, y_train)

# print the best hyperparameter
print('The best hyperparameter: {}'.format(gscv1.best_params_))

# print the best estimator
print('The best estimator: {}'.format(gscv1.best_estimator_))

In [ ]:
# what's the score?
y_hat_gscv1 = gscv1.best_estimator_.predict(X_test)
error = root_mean_squared_error(y_hat_gscv1, y_test)

print('RMSE = {:6f}'.format(error))

**Can it be even better?**

In [ ]:
# HANDS-ON: let's try out a different range for alpha
param_grid2 = ???
gscv2 = ???
???

# print the best hyperparameter
print('The best hyperparameter: {}'.format(gscv2.best_params_))

# print the best estimator
print('The best estimator: {}'.format(gscv2.best_estimator_))

In [ ]:
# what's the score?
y_hat_gscv2 = gscv2.best_estimator_.predict(X_test)
error = root_mean_squared_error(y_hat_gscv2, y_test)

print('RMSE = {:6f}'.format(error))

**Question** \
Does it make sense to go further with hyperparameter search? \
What happened with test set?

In [ ]:
# HANDS-ON: make grid search CV with different splits
cv = ???
gscv = ???
???
print('The best hyperparameter: {}'.format(gscv.best_params_))

### Nested cross validation

<img src="../data/img/nestedCV1.png" alt="nestedCV1" width="650"/>

In [ ]:
# import package for plotting and statistics
from scipy import stats

# Set estimator, parameter grid, scoring
estimator = Ridge()
p_grid = {'alpha': [2, 5, 7, 10, 25, 50, 75]}
scoring = 'neg_root_mean_squared_error'

# Choose cross-validation techniques for the inner and outer loops, independently of the dataset.
outer_cv = ???
inner_cv = ???

# parameter search and scoring
gscv = ???
scores = ???

# print all errors including mean and std
print('RMSE = ')
for e, f in zip(scores, range(1, 6)):
    print('Fold {}: {:.6}'.format(f, -e))
print('Mean  : {:.6}'.format(-np.array(errors).mean()))
print('Std   : {:.6}'.format(np.array(errors).std()))

**Showcase** \
Sklearn example with multiple nested CVs \
https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html

## 2. Simple ensemble methods

1) Averaging
2) Weighted averaging
3) Blending
4) Stacking

### HANDS-ON: Learning base models
- linear regression
- ridge
- lasso
- SVM

In [ ]:
# import stuff
from sklearn.linear_model import Lasso
from sklearn.svm import SVR

#### HANDs-ON: build Linear Regression (LR) model
https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html

In [ ]:
# 1. Create/build model
LR = ???
# 2. Fit model
???
# 3. Evaluate model
y_LR = ???
error_LR = ???
print('Linear regression RMSE: {:6f}'.format(error_LR))

#### HANDS-ON: build Ridge regression model, and tune its hyperparameter
https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html

In [ ]:
# find the best hyperparameter
???
print('The best hyperparameter for Ridge is: {}'.format(gscv.best_params_))

In [ ]:
# 1. Create/build model
???
# 2. Fit model
???
# 3. Evaluate model
???
error_ridge = ???
print('Ridge regression RMSE: {:6f}'.format(error_ridge))

#### HANDS-ON: build Lasso regression model, and tune its hyperparameter
https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html

In [ ]:
# find the best hyperparameter
???

In [ ]:
# 1. Create/build model
???
# 2. Fit model
???
# 3. Evaluate model
???

#### HANDS-ON: build Suport Vector Machine (SVM) model
https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVR.html

In [ ]:
# find the best hyperparameters
estimator = SVR(cache_size=2000)
param_grid = [
    {
        'kernel': ['linear'],
        'C': [0.1, 0.5, 1, 5, 10],
        'epsilon': [0.01, 0.05, 0.1, 0.5, 1]
    },
    {
        'kernel': ['poly'],
        'degree': [2, 3, 4],
        'gamma': ['scale', 'auto'],
        'C': [0.1, 0.5, 1, 5, 10],
        'epsilon': [0.01, 0.05, 0.1, 0.5, 1]
    }, 
    {
        'kernel': ['rbf'],
        'gamma': ['scale', 'auto'],
        'C': [0.1, 0.5, 1, 5, 10],
        'epsilon': [0.01, 0.05, 0.1, 0.5, 1]
    }, 
    {
        'kernel': ['sigmoid'],
        'gamma': ['scale', 'auto'],
        'C': [0.1, 0.5, 1, 5, 10],
        'epsilon': [0.01, 0.05, 0.1, 0.5, 1]
    }
]
gscv = GridSearchCV(estimator, param_grid, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
gscv.fit(X_train, y_train)
print('The best hyperparameter for SVM is: {}'.format(gscv.best_params_))

In [ ]:
# 1. Create/build model
???
# 2. Fit model
???
# 3. Evaluate model
???
print('SVM regression RMSE: {:6f}'.format(error_SVM))

### Averaging

In [ ]:
# HANDS-ON: averaging with no weights
y_avg = ???
error_avg = ???
print('Averaging RMSE: {:6f}'.format(error_avg))

In [ ]:
# HANDS-ON: weighted averaging
y_avg = ???
error_avg = ???
print('Averaging RMSE: {:6f}'.format(error_avg))

### Blending

In [ ]:
# prepare data
X_train_blend, X_val_blend, y_train_blend, y_val_blend = train_test_split(X_train, y_train, test_size=0.5, random_state=123)

**1. Train each model on train set**   
![blending 1.](../data/img/blend1_0.25.png)

In [ ]:
# LR
LR_blend = LinearRegression().fit(X_train_blend, y_train_blend)

# Lasso
lasso_blend = Lasso(alpha=0.001).fit(X_train_blend, y_train_blend)

# Ridge
ridge_blend = Ridge(alpha=10).fit(X_train_blend, y_train_blend)

# SVM
SVM_blend = SVR(C=1, epsilon=0.05, gamma='auto', kernel='rbf').fit(X_train_blend, y_train_blend)

**2. Use trained models to generate training set for metamodel** \
![blending 2.](../data/img/blend2_0.25.png)

In [ ]:
# generate predictions
y_LR_blend = LR_blend.predict(X_val_blend)
y_lasso_blend = lasso_blend.predict(X_val_blend)
y_ridge_blend = ridge_blend.predict(X_val_blend)
y_SVM_blend = SVM_blend.predict(X_val_blend)

In [ ]:
# gather predictions into train set for the metamodel
X_train_meta = pd.DataFrame({
    'y_LR_blend': y_LR_blend,
    'y_lasso_blend': y_lasso_blend,
    'y_ridge_blend': y_ridge_blend,
    'y_SVM_blend': y_SVM_blend
}, index=y_val_blend.index)
X_train_meta.head()

**3. a) Use newly generated data to train meta-model**   
![blending 3.a](../data/img/blend3a_0.25.png)

In [ ]:
# use Ridge with default hyperparameter (Singular Value Decomposition for numerical stability)
meta = Ridge(solver='svd').fit(X_train_meta, y_val_blend)

**3. b) Train each model on the whole train+validation set**   
![blend 3.b](../data/img/blend3b_0.25.png)

In [ ]:
# LR
LR_blend = ???

# Lasso
lasso_blend = ???

# Ridge
ridge_blend = ???

# SVM
SVM_blend = ???

#### Evaluate the blending ensemble   
![evaluate blending](../data/img/blend4_0.25.png)

In [ ]:
# HANDS-ON: use test set to make predictions with base models
y_LR_blend = ???
y_lasso_blend = ???
y_ridge_blend = ???
y_SVM_blend = ???

In [ ]:
# HANDS-ON: gather base predictions
X_test_meta = ???

In [ ]:
# HANDS-ON: make predictions
y_blend = ???

# score of blending
error_blend = root_mean_squared_error(y_test, y_blend)
print('Blending RMSE: {:6f}'.format(error_blend))

### Stacking

https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingRegressor.html

In [ ]:
# import stacking regressor
from sklearn.ensemble import StackingRegressor

In [ ]:
# make stacking ensemble

# HANDS-ON: define estimators as list of tuples, where each tuple consists of a string (name) and a regressor - use previous hyperparameters
estimators = ???

# define stacking regressor
stack = StackingRegressor(estimators, final_estimator=Ridge(solver='svd'), cv=4)

In [ ]:
# fit and predict
stack.fit(X_train, y_train)
y_stack = stack.predict(X_test)

In [ ]:
# score of stacking
error_stack = root_mean_squared_error(y_test, y_stack)
print('Stacking RMSE: {:6f}'.format(error_stack))

## Part 2: Bootstrap aggregating (bagging)   
![bagging](../data/img/bagging_0.25.png)

### Random Forests   

https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html

In [ ]:
# import random forest
from sklearn.ensemble import RandomForestRegressor

**Default hyperparameters**

In [ ]:
# Create model (random_state=123, n_jobs=-1)
RF_default = ???

# Fit and predict
RF_default.fit(X_train, y_train)
y_RF_default = RF_default.predict(X_test)

# Evaluate model
error_RF_default = root_mean_squared_error(y_test, y_RF_default)
print('Default RF RMSE: {:6f}'.format(error_RF_default))

In [ ]:
# why is score so "bad"? -> check for overfitting
error_RF_default_train = ???
print('Train RMSE of default RF: {:6f}'.format(error_RF_default_train))

**Tuned hyperparameters**

In [ ]:
# tune hyperparameters using GridSearchCV - tree defining hyperparameters
estimator = RandomForestRegressor(random_state=123, n_jobs=-1)
param_grid = {
    'max_depth': [5, 10],
    'min_samples_split': [2, 4, 8],
    'min_samples_leaf': [1, 2, 4]
}
gscv1 = GridSearchCV(estimator, param_grid, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1, refit=True)
gscv1.fit(X_train, y_train)
print('The best hyperparameters for RF are: {}'.format(gscv1.best_params_))

In [ ]:
# tune hyperparameters using GridSearchCV - bagging defining hyperparameters
estimator = RandomForestRegressor(random_state=234, n_jobs=-1, **gscv1.best_params_)
param_grid = {
    'max_samples': [0.5, 0.6, 0.7, 0.8, 0.9],
    'max_features': [0.5, 0.6, 0.7, 0.8, 0.9]
}
gscv2 = GridSearchCV(estimator, param_grid, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1, refit=True)
gscv2.fit(X_train, y_train)
print('The best hyperparameters for RF are: {}'.format(gscv2.best_params_))

In [ ]:
# tune hyperparameters using GridSearchCV - finalize with increasing number of trees
estimator = RandomForestRegressor(random_state=345, n_jobs=-1, **gscv1.best_params_, **gscv2.best_params_)
param_grid = {
    'n_estimators': [100, 500, 1000, 2000, 3000],
}
gscv3 = GridSearchCV(estimator, param_grid, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1, refit=True)
gscv3.fit(X_train, y_train)
print('The best hyperparameters for RF are: {}'.format(gscv3.best_params_))

In [ ]:
# Evaluate the final RF model
RF_tuned = RandomForestRegressor(
    random_state=123, 
    n_jobs=-1, 
    max_depth=???,
    min_samples_leaf=???,
    min_samples_split=???,
    max_features=???,
    max_samples=???,
    n_estimators=???
).fit(X_train, y_train)
y_RF_tuned = RF_tuned.predict(X_test)
error_RF_tuned = root_mean_squared_error(y_test, y_RF_tuned)
print('Tuned RF RMSE: {:6f}'.format(error_RF_tuned))

In [ ]:
# check for overfitting of tuned RF
error_RF_tuned_train = ???
print('Train RMSE of tuned RF: {:6f}'.format(error_RF_tuned_train))

**If we let GridSearchCV run for a longer time...**

In [ ]:
'''
%%time
# tune hyperparameters using GridSearchCV
estimator = RandomForestRegressor(random_state=321, n_jobs=-1)
param_grid = {
    'max_depth': [3, 5, 8, None],
    'min_samples_split': [2, 4, 8, 12],
    'min_samples_leaf': [1, 2, 4, 6],
    'max_samples': [0.5, 0.6, 0.7, 0.8, 0.9],
    'max_features': [0.5, 0.6, 0.7, 0.8, 0.9],
    'n_estimators': [100, 500, 1000, 2000]
}
gscv = GridSearchCV(estimator, param_grid, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
gscv.fit(X_train, y_train)
print('The best hyperparameter for RF is: {}'.format(gscv.best_params_))
''';

The best hyperparameter for RF is: {'max_depth': None, 'max_features': 0.5, 'max_samples': 0.9, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 2000}  
CPU times: total: 36.2 s  
Wall time: 2h 12min 53s

In [ ]:
# Evaluate RF model tuned for a longer time
RF_tuned = RandomForestRegressor(
    random_state=321, 
    n_jobs=-1, 
    max_depth=None, 
    min_samples_leaf=1,
    min_samples_split=2,
    max_features=0.5, 
    max_samples=0.9, 
    n_estimators=2000
).fit(X_train, y_train)
y_RF_tuned = RF_tuned.predict(X_test)
error_RF_tuned = root_mean_squared_error(y_test, y_RF_tuned)
print('Tuned RF RMSE: {:6f}'.format(error_RF_tuned))

### How does RF score without scaling of the data?

In [ ]:
# prepare unscaled data
df_full_noscale = pd.concat([df_proc[num_cols], df_cat_1hot], axis=1)
X_train_noscale, X_test_noscale, y_train_noscale, y_test_noscale = train_test_split(
    df_full_noscale, df_full['SalePrice'], test_size=0.2, random_state=123
)

In [ ]:
# repeat the error on scaled data
print('Tuned RF RMSE on scaled data:   {:6f}'.format(error_RF_tuned))

# train and evaluate on unscaled data
RF_noscale = RandomForestRegressor(
    random_state=321, 
    n_jobs=-1, 
    max_depth=None, 
    min_samples_leaf=1,
    min_samples_split=2,
    max_features=0.5, 
    max_samples=0.9, 
    n_estimators=2000
).fit(X_train_noscale, y_train_noscale)
y_RF_noscale = ???
error_RF_noscale = ???
print('Tuned RF RMSE on unscaled data: {:6f}'.format(error_RF_noscale))

**For comparison, evaluate score of SVM with and without scaling**

In [ ]:
# With scaling (as before)
print('SVM regression RMSE with scaling:    {:6f}'.format(error_SVM))

# Without scaling
SVM_noscale = SVR(C=1, epsilon=0.05, gamma='auto', kernel='rbf')
SVM_noscale.fit(???)
y_SVM_noscale = SVM_noscale.predict(???)
error_SVM_noscale = root_mean_squared_error(???)
print('SVM regression RMSE without scaling: {:6f}'.format(error_SVM_noscale))

## Part 3: Gradient Boosting Decision Trees

https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingRegressor.html

In [ ]:
# import GBDT
from sklearn.ensemble import GradientBoostingRegressor

**Default hyperparameters**

In [ ]:
# Create/build model (random_state=123)
GBR_default = ???
# Fit model
GBR_default.fit(X_train, y_train)
# Evaluate model
y_GBR_default = GBR_default.predict(X_test)
error_GBR_default = root_mean_squared_error(y_test, y_GBR_default)
print('Default GBR RMSE: {:6f}'.format(error_GBR_default))

**Tuning hyperparameters**

In [ ]:
# 1.) Fix everything except learning_rate and n_estimators
estimator = GradientBoostingRegressor(random_state=234)
param_grid = {
    'learning_rate': [0.1, 0.15, 0.2],
    'n_estimators': [50, 70, 90]
}
gscv1 = GridSearchCV(estimator, param_grid, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1, refit=True)
gscv1.fit(X_train, y_train)
print('The best hyperparameters for GBR are: {}'.format(gscv1.best_params_))

In [ ]:
# 2.) With fixed learning_rate and n_estimators tune tree-specific hyperparameters
estimator = GradientBoostingRegressor(random_state=345, **gscv1.best_params_)
param_grid = {
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5],
    'max_depth': [3, 5, 7]
}
gscv2 = GridSearchCV(estimator, param_grid, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1, refit=True)
gscv2.fit(X_train, y_train)
print('The best hyperparameters for GBR are: {}'.format(gscv2.best_params_))

In [ ]:
# 3.) With fixed already defined hyperparameters tune bagging hyperparameters
estimator = GradientBoostingRegressor(random_state=456, **gscv1.best_params_, **gscv2.best_params_)
param_grid = {
    'subsample': [0.5, 0.6, 0.8, 1],
    'max_features': [0.5, 0.6, 0.8, 1]
}
gscv3 = GridSearchCV(estimator, param_grid, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1, refit=True)
gscv3.fit(X_train, y_train)
print('The best hyperparameters for GBR are: {}'.format(gscv3.best_params_))

In [ ]:
# 4.) Deacrease learning_rate and increase n_estimators
estimator = GradientBoostingRegressor(random_state=567, **gscv1.best_params_, **gscv2.best_params_, **gscv3.best_params_)
param_grid = [
    {
        'learning_rate': [0.1],
        'n_estimators': [100, 200, 300]
    }, 
    {
        'learning_rate': [0.05],
        'n_estimators': [200, 400, 600]
    }, 
    {
        'learning_rate': [0.025],
        'n_estimators': [400, 800, 1200]
    }, 
    {
        'learning_rate': [0.01],
        'n_estimators': [800, 1600, 2400]
    }
]
gscv4 = GridSearchCV(estimator, param_grid, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1, refit=True)
gscv4.fit(X_train, y_train)
print('The best hyperparameters for GBR are: {}'.format(gscv4.best_params_))

In [ ]:
# Evaluate the final model
GBR_tuned = GradientBoostingRegressor(
    learning_rate=0.025,
    n_estimators=1200,
    min_samples_split=2,
    min_samples_leaf=1,
    max_depth=3,
    subsample=1,
    max_features=0.6,
    random_state=123
).fit(X_train, y_train)
y_GBR_tuned = GBR_tuned.predict(X_test)
error_GBR_tuned = root_mean_squared_error(y_test, y_GBR_tuned)
print('Tuned GBR RMSE: {:6f}'.format(error_GBR_tuned))

In [ ]:
# Evaluate the final model - even better
GBR_tuned = GradientBoostingRegressor(
    learning_rate=0.025,
    n_estimators=1200,
    min_samples_split=2,
    min_samples_leaf=1,
    max_depth=3,
    subsample=0.5,
    max_features=0.6,
    random_state=123
).fit(X_train, y_train)
y_GBR_tuned = GBR_tuned.predict(X_test)
error_GBR_tuned = root_mean_squared_error(y_test, y_GBR_tuned)
print('Tuned GBR RMSE: {:6f}'.format(error_GBR_tuned))

### XGBoost

In [ ]:
# import XGBoost
import xgboost as xgb

**Sklearn way** \
https://xgboost.readthedocs.io/en/stable/python/sklearn_estimator.html

In [ ]:
# Create/build model with default hyperparameters (random_state=123)
XGB_default = ???
# Fit model
???
# Fit model
y_XGB_default = ???
error_XGB_default = ???
print('Default XGB RMSE: {:6f}'.format(error_XGB_default))

In [ ]:
# Create/build model with tuned hyperparameters - use previous hyperparameters, check API reference if needed
XGB_tuned = ???
# Fit model
XGB_tuned.fit(X_train, y_train)
# Evaluate model
y_XGB_tuned = XGB_tuned.predict(X_test)
error_XGB_tuned = root_mean_squared_error(y_test, y_XGB_tuned)
print('Tuned XGB RMSE: {:6f}'.format(error_XGB_tuned))

In [ ]:
# Create/build model with tuned hyperparameters, without regularization
XGB_tuned = xgb.XGBRegressor(
    ???
)
# Fit model
XGB_tuned.fit(X_train, y_train)
# Evaluate model
y_XGB_tuned = XGB_tuned.predict(X_test)
error_XGB_tuned = root_mean_squared_error(y_test, y_XGB_tuned)
print('Tuned XGB RMSE: {:6f}'.format(error_XGB_tuned))

**XGBoost with histogram-based algorithm** \
https://xgboost.readthedocs.io/en/latest/python/python_api.html

In [ ]:
# create dataset for XGBoost
xgb_train = xgb.DMatrix(X_train, y_train)
xgb_test = xgb.DMatrix(X_test, y_test)

In [ ]:
# define parameters
xgb_params = {
    'objective': 'reg:squarederror',
    'tree_method': 'hist',
    'learning_rate': 0.025,
    #'n_estimators': 1200,
    'max_depth': 3,
    'subsample': 0.5,
    'colsample_bytree': 0.6,
    'reg_lambda': 0,
    'reg_alpha': 0,
    'random_state': 123,
    'verbosity': 0
}

# train the model
XGB_tuned = xgb.train(xgb_params, xgb_train, num_boost_round=1200)

# evaluate the model
y_XGB_tuned = XGB_tuned.predict(xgb_test)
error_XGB_tuned = root_mean_squared_error(y_test, y_XGB_tuned)
print('RMSE of tuned XGB with histogram-based algorithm: {:6f}'.format(error_XGB_tuned))

In [ ]:
# define parameters
xgb_params['tree_method'] = 'exact'

# train the model
XGB_tuned = xgb.train(xgb_params, xgb_train, num_boost_round=1200)

# evaluate the model
y_XGB_tuned = XGB_tuned.predict(xgb_test)
error_XGB_tuned = root_mean_squared_error(y_test, y_XGB_tuned)
print('RMSE of tuned XGB with exact algorithm: {:6f}'.format(error_XGB_tuned))

**XGBoost with cross-validation**

In [ ]:
# make Dmatrix
xgb_all_data = xgb.DMatrix(X, y)

In [ ]:
xgb.cv(
    xgb_params, 
    xgb_all_data, 
    num_boost_round=1200, 
    nfold=5, 
    metrics=('rmse', 'mae'), 
    # show_stdv=True,
    seed=1
)

### LightGBM

LightGBM Python API: https://lightgbm.readthedocs.io/en/latest/Python-API.html \
LightGBM parameters: https://lightgbm.readthedocs.io/en/stable/Parameters.html \
LightGBM parameters tuning: https://lightgbm.readthedocs.io/en/stable/Parameters-Tuning.html

In [ ]:
# import LightGBM
import lightgbm as lgb

**Sklearn API** \
https://lightgbm.readthedocs.io/en/latest/pythonapi/lightgbm.LGBMRegressor.html

In [ ]:
# Create/build model with default parameters (random_state=123)
LGBM_default = lgb.LGBMRegressor(random_state=123)
# Fit model
LGBM_default.fit(X_train, y_train)
# Evaluate model
y_LGBM_default = LGBM_default.predict(X_test)
error_LGBM_default = root_mean_squared_error(y_test, y_LGBM_default)
print('Default LGBM RMSE: {:6f}'.format(error_LGBM_default))

In [ ]:
# Create/build model with tuned hyperparameters
LGBM_tuned = lgb.LGBMRegressor(
    learning_rate=0.015,
    n_estimators=2000,
    min_child_samples=1,
    num_leaves=9,
    max_depth=5,
    subsample=0.5,
    subsample_freq=1,
    colsample_bytree=0.6,
    random_state=123,
    verbose=-1
)
# Fit model
LGBM_tuned.fit(X_train, y_train)
# Evaluate model
y_LGBM_tuned = LGBM_tuned.predict(X_test)
error_LGBM_tuned = root_mean_squared_error(y_test, y_LGBM_tuned)
print('Tuned LGBM RMSE: {:6f}'.format(error_LGBM_tuned))

**LightGBM with early stopping** \
https://lightgbm.readthedocs.io/en/latest/Parameters.html

In [ ]:
# create dataset for lightgbm
X_train_lgbm, X_val_lgbm, y_train_lgbm, y_val_lgbm = train_test_split(X_train, y_train, test_size=0.2, random_state=321)
lgb_train = lgb.Dataset(X_train_lgbm, y_train_lgbm, free_raw_data=False)
lgb_val = lgb.Dataset(X_val_lgbm, y_val_lgbm, reference=lgb_train, free_raw_data=False)

In [ ]:
# define parameters
params = {
    'boosting': 'gbdt',
    'objective': 'regression',
    'metric': 'root_mean_squared_error',
    'learning_rate': 0.015,
    'min_child_samples': 1,
    'num_leaves': 9,
    'max_depth': 5,
    'subsample': 0.5,
    'subsample_freq': 1,
    'colsample_bytree': 0.6,
    'random_state': 123,
    'verbose': 0
}

In [ ]:
# train the model
lgbm = lgb.train(
    params,
    lgb_train,
    num_boost_round=2000,
    valid_sets=lgb_val,
    callbacks=[
        lgb.early_stopping(stopping_rounds=100),
    ]
)

In [ ]:
# retrain the model on the full train set
lgb_train = lgb.Dataset(X_train, y_train, free_raw_data=False)
lgbm = lgb.train(
    params,
    lgb_train,
    num_boost_round=1200,
)

# evaluate the model
y_lgbm = lgbm.predict(X_test)
error_lgbm = root_mean_squared_error(y_test, y_lgbm)
print('RMSE of LGBM with early stopping: {:6f}'.format(error_lgbm))

**Feature importance**

In [ ]:
# feature importance
lgb.plot_importance(lgbm, figsize=(10,12));

**LightGBM with categorical features** \
https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.Dataset.html

In [ ]:
# check the data
print(df_proc.dtypes)
df_proc.head()

In [ ]:
# prepare data
df_no1hot = df_proc.copy()

# scale numerical features
df_no1hot[num_cols] = scaler.transform(df_proc[num_cols])

# change dtypes of categorical features into 'category'
df_no1hot[cat_cols] = df_no1hot[cat_cols].astype('category')

# split the data
X_train_no1hot, X_test_no1hot, y_train_no1hot, y_test_no1hot = train_test_split(
    df_no1hot.drop(columns='SalePrice'), df_no1hot['SalePrice'], test_size=0.2, random_state=123
)

# check if the split is the same as before - check labels
print((y_train==y_train_no1hot).sum())
print((y_test==y_test_no1hot).sum())

In [ ]:
# check the newly prepared data
print(X_train_no1hot.dtypes)
X_train_no1hot.head()

In [ ]:
# create dataset for lightgbm
lgb_train_cat = lgb.Dataset(X_train_no1hot, y_train_no1hot, free_raw_data=False)
lgb_test_cat = lgb.Dataset(X_test_no1hot, y_test_no1hot, reference=lgb_train_cat, free_raw_data=False)

In [ ]:
# train the model
lgbm_cat = lgb.train(
    params,
    lgb_train_cat,
    num_boost_round=1200,
)

# evaluate the model
y_lgbm_cat = lgbm_cat.predict(X_test_no1hot)
error_lgbm_cat = root_mean_squared_error(y_test_no1hot, y_lgbm_cat)
print('RMSE of LGBM with categorical features: {:6f}'.format(error_lgbm_cat))

In [ ]:
lgb.plot_importance(lgbm_cat, figsize=(8,5));

In [ ]:
# plotting a tree
lgb.plot_tree(lgbm_cat, tree_index=0, figsize=(12,10));

**Extra exercise:** tune number of trees (use early stopping) according to data with categorical features.

### CatBoost
CatBoost documentation: https://catboost.ai/en/docs/

In [ ]:
from catboost import Pool, CatBoostRegressor

In [ ]:
# create dataset for catboost
train_pool = Pool(X_train_no1hot, 
                  y_train_no1hot, 
                  cat_features=cat_cols)
test_pool = Pool(X_test_no1hot,
                 y_test_no1hot,
                 cat_features=cat_cols)

In [ ]:
# define the model and train - default hyperparameters
CBR_default = CatBoostRegressor(random_state=123, verbose=0)
CBR_default.fit(train_pool)

# evaluate the model
y_CBR_default = CBR_default.predict(X_test_no1hot)
error_CBR_default = root_mean_squared_error(y_test_no1hot, y_CBR_default)
print('RMSE of CatBoost with default hyperparameters: {:6f}'.format(error_CBR_default))

In [ ]:
# define the model and train - hyperparameters as GBR
CBR_asGBR = CatBoostRegressor(
    objective='RMSE',
    learning_rate=0.025,
    n_estimators=1200,
    max_depth=3,
    subsample=0.5,
    colsample_bylevel=0.6,
    random_state=123,
    verbose=0
)
CBR_asGBR.fit(train_pool)

# evaluate the model
y_CBR_asGBR = CBR_asGBR.predict(X_test_no1hot)
error_CBR_asGBR = root_mean_squared_error(y_test_no1hot, y_CBR_asGBR)
print('RMSE of CatBoost with GBR hyperparameters: {:6f}'.format(error_CBR_asGBR))

In [ ]:
# define the model and train - hyperparameters as XGB
CBR_asXGB = CatBoostRegressor(
    objective='RMSE',
    learning_rate=0.025,
    n_estimators=1200,
    max_depth=3,
    subsample=0.5,
    colsample_bylevel=0.6,
    reg_lambda=0,
    random_state=123,
    verbose=0
)
CBR_asXGB.fit(train_pool)

# evaluate the model
y_CBR_asXGB = CBR_asXGB.predict(X_test_no1hot)
error_CBR_asXGB = root_mean_squared_error(y_test_no1hot, y_CBR_asXGB)
print('RMSE of CatBoost with XGBoost hyperparameters: {:6f}'.format(error_CBR_asXGB))

In [ ]:
# define the model and train - hyperparameters as LightGBM
CBR_asLight = CatBoostRegressor(
    objective='RMSE',
    eval_metric='RMSE',
    iterations=1200,
    learning_rate=0.015,
    min_child_samples=1,
    max_depth=5,
    subsample=0.5,
    colsample_bylevel=0.6,
    reg_lambda=0,
    early_stopping_rounds=200,
    random_state=123,
    verbose=0
)
CBR_asLight.fit(train_pool, eval_set=test_pool)

# evaluate the model
y_CBR_asLight = CBR_asLight.predict(X_test_no1hot)
error_CBR_asLight = root_mean_squared_error(y_test_no1hot, y_CBR_asLight)
print('RMSE of CatBoost with LightGBM hyperparameters: {:6f}'.format(error_CBR_asLight))

**CatBoost plots**

In [ ]:
# plotting a tree
CBR_asLight.plot_tree(0)

In [ ]:
# plotting feature importance
feature_importance = CBR_asLight.feature_importances_
sorted_idx = np.argsort(feature_importance)
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(range(len(sorted_idx)), feature_importance[sorted_idx], align='center')
ax.bar_label(bars)
plt.yticks(range(len(sorted_idx)), np.array(X_test_no1hot.columns)[sorted_idx])
plt.title('Feature Importance');

# All done!

### Bonus: ensemble of ensemble

_University of Zagreb Faculty of Electrical Engineering and Computing_  
_Laboratory for Renewable Energy Systems_  

_Course: AI bootcamp - Foundations of AI_ \
_Notebook: 3b_Ensemble_Learning_

_Website: [www.lares.fer.hr](https://www.lares.fer.hr/)_  
_Contact: [filip.rukavina@fer.hr](mailto:filip.rukavina@fer.hr)_